# motif-discover: Fast de novo DNA Motif Discovery

**Comparable accuracy to MEME, 15× faster than STREME.**

This notebook demonstrates motif-discover on 11 ENCODE K562 ChIP-seq transcription factor profiles, then shows benchmark results from the full 132-TF comparison.

### Headline Results (132 ENCODE TFs)

| Metric | motif-discover | STREME | MEME |
|--------|---------------|--------|------|
| AUROC (shuffled neg.) | **0.842** | 0.803 | — |
| AUROC (genomic neg.) | **0.891** | 0.863 | 0.881 |
| Time per TF | **0.3s** | 3.3s | ~90s |
| Wilcoxon p vs STREME | — | 4.1×10⁻⁷ | — |

---

## 1. Setup

Download the binary and example data. This notebook is designed for Google Colab (Linux x86-64).

In [ ]:
# Clone the repo (contains binary, example data, and benchmark CSVs)
!git clone -q https://github.com/Travis42/motif-discover.git
%cd motif-discover
!chmod +x motif-discover

import os
print(f'Example data: {len([f for f in os.listdir("example") if f.endswith(".fa")])} TFs')
!ls -la motif-discover
!file motif-discover

## 2. Run Motif Discovery

Discover motifs across all example TFs. The binary processes 500bp peak-centered sequences: it extracts the central 100bp for motif discovery and evaluates AUROC on the full 500bp sequences against dinucleotide-shuffled negatives.

In [ ]:
%%time
import subprocess, time

t0 = time.time()
result = subprocess.run(['./motif-discover', '--data', 'example/', '--ours-only', '--no-meme'],
                      capture_output=True, text=True)
elapsed = time.time() - t0
print(result.stderr if result.stderr else result.stdout)

In [ ]:
import pandas as pd
import io

# Parse TSV output
output = result.stderr + '\n' + result.stdout
data_lines = [l for l in output.split('\n') if '\t' in l and 'ours' in l and not l.startswith('TF')]

rows = []
for line in data_lines:
    parts = line.split('\t')
    tf = parts[0].split('...')[-1].strip() if '...' in parts[0] else parts[0]
    rows.append({
        'TF': tf,
        'Width': int(parts[1]),
        'AUROC': float(parts[2]),
        'Time_s': float(parts[3]),
        'P_value': parts[5] if len(parts) > 5 else '',
    })

df = pd.DataFrame(rows)
print(f'Ran {len(df)} TFs in {elapsed:.1f}s ({elapsed/len(df):.2f}s/TF)')
df[['TF', 'Width', 'AUROC', 'Time_s']]

In [ ]:
# Summary statistics
print(f'Average AUROC:  {df["AUROC"].mean():.4f}')
print(f'Median  AUROC:   {df["AUROC"].median():.4f}')
print(f'Average Time:   {df["Time_s"].mean():.2f}s')
print(f'Total Time:     {df["Time_s"].sum():.1f}s')
print(f'TFs with AUROC >= 0.8: {(df["AUROC"] >= 0.8).sum()}/{len(df)}')

## 3. Results Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: AUROC per TF
sorted_df = df.sort_values('AUROC', ascending=True)
colors = ['#2ca02c' if a >= 0.8 else '#ff7f0e' if a >= 0.7 else '#d62728' for a in sorted_df['AUROC']]

axes[0].barh(range(len(sorted_df)), sorted_df['AUROC'], color=colors, edgecolor='black', linewidth=0.3)
axes[0].set_yticks(range(len(sorted_df)))
axes[0].set_yticklabels(sorted_df['TF'], fontsize=9)
axes[0].set_xlabel('AUROC')
axes[0].set_title('Motif Discovery Quality per TF')
axes[0].axvline(0.5, color='gray', linestyle='--', alpha=0.3, label='Random (0.5)')
axes[0].legend()

# Panel B: Time per TF
axes[1].bar(range(len(sorted_df)), sorted_df['Time_s'], color='#457b9d', edgecolor='black', linewidth=0.3)
axes[1].set_xticks(range(len(sorted_df)))
axes[1].set_xticklabels(sorted_df['TF'], rotation=45, ha='right', fontsize=9)
axes[1].set_ylabel('Time (seconds)')
axes[1].set_title('Runtime per TF')

plt.tight_layout()
plt.show()

## 4. Full Benchmark Comparison: motif-discover vs STREME

Pre-computed results from the full 132-TF benchmark (ENCODE K562 ChIP-seq profiles).

- **Shuffled negatives**: dinucleotide-shuffled positives (preserves composition, destroys motif order)
- **132 TFs**: Full ENCODE K562 transcription factor set

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import wilcoxon

plt.rcParams['figure.dpi'] = 120

# Load benchmark data
benchmark_url = 'https://raw.githubusercontent.com/Travis42/motif-discover/main/benchmark_data/shuffled_negatives_132tf.csv'
try:
    shuf = pd.read_csv(benchmark_url)
except:
    # Fallback for local runs
    shuf = pd.read_csv('benchmark_data/shuffled_negatives_132tf.csv')

common = shuf.dropna(subset=['motif_discover_AUROC', 'STREME_AUROC'])

md_auroc = common['motif_discover_AUROC'].mean()
st_auroc = common['STREME_AUROC'].mean()
md_time = common['motif_discover_Time_s'].mean()
st_time = common['STREME_Time_s'].mean()

print(f'=== 132-TF Shuffled Negatives Benchmark ===')
print(f'motif-discover:  AUROC={md_auroc:.4f}  ({md_time:.2f}s/TF)')
print(f'STREME:          AUROC={st_auroc:.4f}  ({st_time:.2f}s/TF)')
print(f'Δ AUROC:         +{md_auroc - st_auroc:.4f}')
print(f'Speedup:         {st_time/md_time:.1f}×')

In [ ]:
# Scatter plot: motif-discover vs STREME, per TF
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(common['STREME_AUROC'], common['motif_discover_AUROC'], alpha=0.5, s=25, c='#2E86AB')
ax.plot([0.4, 1.0], [0.4, 1.0], 'k--', alpha=0.3, label='y = x (tie)')
ax.set_xlabel('STREME AUROC', fontsize=12)
ax.set_ylabel('motif-discover AUROC', fontsize=12)
ax.set_title(f'motif-discover vs STREME ({len(common)} TFs, shuffled neg.)', fontsize=13)

# Annotate outliers
for _, row in common.iterrows():
    d = row['motif_discover_AUROC'] - row['STREME_AUROC']
    if abs(d) > 0.15:
        ax.annotate(row['TF'], (row['STREME_AUROC'], row['motif_discover_AUROC']), fontsize=8)

ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Statistical test
diffs = common['motif_discover_AUROC'] - common['STREME_AUROC']
stat, p = wilcoxon(diffs)
wins = (diffs > 0.001).sum()
losses = (diffs < -0.001).sum()
ties = len(diffs) - wins - losses

print(f'Wilcoxon signed-rank test (n={len(common)}):')
print(f'  p-value   = {p:.2e}')
print(f'  motif-discover wins: {wins}')
print(f'  STREME wins:         {losses}')
print(f'  Ties:                {ties}')
print(f'  Mean Δ AUROC:        +{diffs.mean():.4f}')
print(f'  Median Δ AUROC:      +{diffs.median():.4f}')
print(f'\n  → The improvement is statistically significant (p < 0.001).')

In [ ]:
# Speed comparison chart
fig, ax = plt.subplots(figsize=(7, 5))

tools = ['motif-discover', 'STREME']
times = [md_time, st_time]
colors = ['#2E86AB', '#E63946']

bars = ax.bar(tools, times, color=colors, edgecolor='black', linewidth=0.5, width=0.5)
ax.set_ylabel('Time per TF (seconds)', fontsize=12)
ax.set_title('Speed: motif-discover vs STREME (132-TF benchmark)', fontsize=13)

for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{t:.2f}s', ha='center', fontsize=13, fontweight='bold')

ax.text(0.5, max(times) * 0.85, f'{st_time/md_time:.0f}× faster', ha='center', fontsize=14, color='#2E86AB', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Genomic Negatives Benchmark (Realistic)

Using random hg38 genomic regions as negatives (harder than shuffled). Markov-1 background scoring. Available for the 53-TF subset where STREME and MEME genomic results exist.

| Tool | Mean AUROC | Time/TF |
|------|-----------|---------|
| **motif-discover** | **0.891** | **0.3s** |
| MEME | 0.881 | ~90s |
| STREME | 0.863 | 4.4s |

In [ ]:
# Genomic comparison chart
genomic_url = 'https://raw.githubusercontent.com/Travis42/motif-discover/main/benchmark_data/genomic_negatives_132tf.csv'
try:
    gen = pd.read_csv(genomic_url)
except:
    gen = pd.read_csv('benchmark_data/genomic_negatives_132tf.csv')

common_g = gen.dropna(subset=['motif_discover_AUROC', 'STREME_AUROC'])

fig, ax = plt.subplots(figsize=(8, 5))

tools_g = ['motif-discover', 'STREME']
if 'MEME_AUROC' in common_g.columns:
    meme_common = common_g.dropna(subset=['MEME_AUROC'])
    tools_g.append('MEME')
    aurocs_g = [
        common_g['motif_discover_AUROC'].mean(),
        common_g['STREME_AUROC'].mean(),
        meme_common['MEME_AUROC'].mean(),
    ]
    colors_g = ['#2E86AB', '#E63946', '#F4A261']
else:
    aurocs_g = [common_g['motif_discover_AUROC'].mean(), common_g['STREME_AUROC'].mean()]
    colors_g = ['#2E86AB', '#E63946']

bars = ax.bar(tools_g, aurocs_g, color=colors_g, edgecolor='black', linewidth=0.5, width=0.5)
ax.set_ylabel('Mean AUROC', fontsize=12)
ax.set_title(f'Genomic Negatives ({len(common_g)} TFs, Markov-1 scoring)', fontsize=13)
ax.set_ylim(0.82, 0.92)

for bar, a in zip(bars, aurocs_g):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001, f'{a:.4f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Try Your Own Data

Provide FASTA files named `<TF>_sequences.fa` (500bp peak-centered) and run:

```bash
./motif-discover --data your_data/ --ours-only
```

The binary is statically linked — no dependencies, no installation. Works on any x86-64 Linux system.

**Input format:** FASTA files with ~200 sequences per TF, each 500bp, centered on ChIP-seq peak summits.

---

## Citation

Paper in preparation. For questions: travis.smith42@pm.me